# Kaitu AI: Phi-4-mini QLoRA Fine-Tuning on Kaggle GPU
Automated background training job dispatched from Kaitu fine-tuning pipeline.

In [ ]:
# 1. Environment & GPU Verification
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!nvidia-smi

In [ ]:
# 2. Install Required Libraries & Avoid Conflicts
!pip uninstall -y torchvision torchao
!pip install -q -U 'transformers>=4.46.0,<4.49.0' peft trl bitsandbytes accelerate gguf
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False

In [ ]:
# 3. Decode & Write Embedded Datasets
import base64

train_b64 = '...'
eval_b64 = '...'

with open('train.jsonl', 'wb') as f:
    f.write(base64.b64decode(train_b64))

with open('eval.jsonl', 'wb') as f:
    f.write(base64.b64decode(eval_b64))

print('Dataset files decoded successfully into workspace.')

In [ ]:
# 4. Load Base Model in 4-bit NF4 Quantization
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import sys
import torch
import transformers
import transformers.utils
import transformers.modeling_utils
import transformers.processing_utils

# Polyfills for Microsoft Phi-4 remote code compatibility
if not hasattr(transformers, 'modeling_attn_mask_utils') or 'transformers.modeling_attn_mask_utils' not in sys.modules:
    try:
        import transformers.masking_utils as masking_utils
        sys.modules['transformers.modeling_attn_mask_utils'] = masking_utils
        transformers.modeling_attn_mask_utils = masking_utils
    except Exception:
        pass

try:
    from typing import TypedDict
except Exception:
    from typing_extensions import TypedDict

if not hasattr(transformers.utils, 'LossKwargs'):
    class LossKwargs(TypedDict, total=False):
        pass
    transformers.utils.LossKwargs = LossKwargs

if not hasattr(transformers.modeling_utils, 'ALL_ATTENTION_FUNCTIONS'):
    transformers.modeling_utils.ALL_ATTENTION_FUNCTIONS = {}

if not hasattr(transformers.processing_utils, 'Unpack'):
    try:
        from typing_extensions import Unpack
    except Exception:
        try:
            from typing import Unpack
        except Exception:
            class Unpack:
                def __class_getitem__(cls, item): return item
    transformers.processing_utils.Unpack = Unpack

from transformers.modeling_utils import PreTrainedModel
if hasattr(PreTrainedModel, 'get_expanded_tied_weights_keys'):
    orig_get_expanded = PreTrainedModel.get_expanded_tied_weights_keys
    def safe_get_expanded_tied_weights_keys(self, all_submodels=False):
        if isinstance(getattr(self, '_tied_weights_keys', None), list):
            self._tied_weights_keys = {k: 'model.embed_tokens.weight' for k in self._tied_weights_keys}
        try:
            return orig_get_expanded(self, all_submodels=all_submodels)
        except Exception:
            return self._tied_weights_keys if isinstance(getattr(self, '_tied_weights_keys', None), dict) else {}
    PreTrainedModel.get_expanded_tied_weights_keys = safe_get_expanded_tied_weights_keys

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

model_id = 'microsoft/Phi-4-mini-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True
)

# Materialize unallocated meta buffers in rotary embeddings before forward pass
for module in model.modules():
    if hasattr(module, 'original_inv_freq'):
        dim = getattr(module, 'dim', 96)
        base = getattr(module, 'rope_theta', 10000.0)
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32, device='cuda') / dim))
        if hasattr(module, '_buffers'):
            module._buffers['original_inv_freq'] = inv_freq
            module._buffers['inv_freq'] = inv_freq.clone()
        module.__dict__['original_inv_freq'] = inv_freq
        module.__dict__['inv_freq'] = inv_freq.clone()
    if hasattr(module, '_longrope_frequency_update'):
        def make_update(mod):
            def update(position_ids, device):
                dim = getattr(mod, 'dim', 96)
                base = getattr(mod, 'rope_theta', 10000.0)
                inv = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32, device=device) / dim))
                if hasattr(mod, '_buffers'):
                    mod._buffers['original_inv_freq'] = inv
                    mod._buffers['inv_freq'] = inv.clone()
                mod.__dict__['original_inv_freq'] = inv
                mod.__dict__['inv_freq'] = inv.clone()
            return update
        module._longrope_frequency_update = make_update(module)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
# 5. Train Model with Standard Hugging Face Trainer
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from transformers import Trainer

dataset = load_dataset('json', data_files={'train': 'train.jsonl', 'eval': 'eval.jsonl'})

# Format and tokenize into native Phi-4 chat tokens with completion-only loss masking
def format_and_tokenize(batch):
    all_input_ids = []
    all_labels = []
    all_masks = []
    for msgs in batch['messages']:
        prompt_str = ''
        full_str = ''
        for m in msgs:
            r = m['role']
            c = m['content']
            piece = f'<|{r}|>\n{c}<|end|>\n'
            if r != 'assistant':
                prompt_str += piece
            full_str += piece
        if not prompt_str.endswith('<|assistant|>\n'):
            prompt_str += '<|assistant|>\n'

        prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        full_ids = tokenizer.encode(full_str, add_special_tokens=False)
        if len(full_ids) > 1024:
            full_ids = full_ids[:1024]

        labels = [-100] * len(full_ids)
        p_len = min(len(prompt_ids), len(full_ids))
        for i in range(p_len, len(full_ids)):
            labels[i] = full_ids[i]

        all_input_ids.append(full_ids)
        all_labels.append(labels)
        all_masks.append([1] * len(full_ids))

    return {'input_ids': all_input_ids, 'labels': all_labels, 'attention_mask': all_masks}

tokenized_dataset = dataset.map(format_and_tokenize, batched=True, remove_columns=dataset['train'].column_names)

from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors='pt',
    padding=True
)

training_args = TrainingArguments(
    output_dir='./checkpoints',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1.8e-4,
    lr_scheduler_type='cosine',
    warmup_steps=25,
    weight_decay=0.01,
    num_train_epochs=3,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['eval'],
    data_collator=data_collator,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()
adapter_path = './kaitu-phi4-mini-adapter'
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print('LoRA adapter saved successfully.')

In [ ]:
# 6. Merge LoRA Weights into Base Model (CPU/BF16 to save VRAM)
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
from peft import PeftModel
import gc

del model
del trainer
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    trust_remote_code=True
)
merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

merged_path = './kaitu-merged'
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print('Merged model saved successfully.')

In [ ]:
# 7. Convert and Quantize to Q4_K_M GGUF
import os
import glob
import shutil
import subprocess
from huggingface_hub import snapshot_download

print('[+] Cloning llama.cpp...')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'], check=True)
subprocess.run(['pip', 'install', '-q', 'gguf', 'protobuf'], check=True)

print('[+] Downloading tokenizer metadata into merged model...')
snapshot_download('microsoft/Phi-4-mini-instruct', allow_patterns=['tokenizer*', 'vocab*', 'merges*'], local_dir='./kaitu-merged')

print('[+] Patching llama.cpp/conversion/phi.py for BPE tokenizer...')
phi_file = 'llama.cpp/conversion/phi.py'
with open(phi_file, 'r', encoding='utf-8') as f:
    code = f.read()

target = "raise ValueError(f'Error: Missing {tokenizer_path}')"
if target in code:
    code = code.replace(target, 'return self._set_vocab_gpt2()')
    with open(phi_file, 'w', encoding='utf-8') as f:
        f.write(code)
    print('Patched llama.cpp/conversion/phi.py successfully!')
else:
    print('Target string not found in phi.py, checking fallback...')

print('[+] Converting to F16 GGUF...')
subprocess.run(['python3', 'llama.cpp/convert_hf_to_gguf.py', './kaitu-merged', '--outfile', './kaitu-f16.gguf', '--outtype', 'f16'], check=True)

# Free unneeded weights to guarantee plenty of disk space for quantization
for f in glob.glob('./kaitu-merged/*.safetensors') + glob.glob('./checkpoints/*'):
    try:
        os.remove(f)
    except Exception:
        pass

print('[+] Compiling llama-quantize...')
subprocess.run('cd llama.cpp && (cmake -B build -DGGML_CUDA=OFF && cmake --build build --config Release -j --target llama-quantize || make -j llama-quantize)', shell=True, check=True)

quant_bin = None
for root, dirs, files in os.walk('llama.cpp'):
    if 'llama-quantize' in files:
        p = os.path.join(root, 'llama-quantize')
        if os.access(p, os.X_OK):
            quant_bin = p
            break

if not quant_bin:
    raise RuntimeError('llama-quantize binary not found!')

out_gguf = '/kaggle/working/kaitu-phi4-mini-q4_k_m.gguf'
print(f'[+] Quantizing to {out_gguf} (Q4_K_M)...')
subprocess.run([quant_bin, './kaitu-f16.gguf', out_gguf, 'Q4_K_M'], check=True)

# Clean up intermediate files
for p in ['./kaitu-f16.gguf', './kaitu-merged', './llama.cpp']:
    if os.path.isfile(p):
        os.remove(p)
    elif os.path.isdir(p):
        shutil.rmtree(p, ignore_errors=True)

print(f'SUCCESS: Output GGUF ready at {out_gguf} ({os.path.getsize(out_gguf)} bytes)')